# Plot Inference Results with Different Sample Sizes

Run `../script/run_1.sh` first. This generates inference results for synthetic datasets with different sample sizes, where sample size is the number of NK cells included in the analysis.

The goal is to assess how many NK cells are needed to recover the input parameters with reasonable accuracy.

This notebook loads the saved inference results from the latest `../results/part_1_runs/...` folder and makes plots.

In [34]:
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

import seaborn as sns
from matplotlib import gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import MultipleLocator, MaxNLocator, NullLocator

try:
    import scienceplots
    plt.style.use("science")
except ImportError:
    pass

LATEST_RUN_FILE = Path("../results/part_1_latest.txt")
RUN_NAME = LATEST_RUN_FILE.read_text().strip() if LATEST_RUN_FILE.exists() else "part_1"
RESULTS_DIR = Path("../results") / RUN_NAME

PARAM_LABELS = {
    "mu_lambda": r"$\mu_\lambda$",
    "sigma_lambda": r"$\sigma_\lambda$",
    "p_zero": r"$\phi_0$",
}

PARAM_XLIMS = {
    "mu_lambda": (0, 8),
    "sigma_lambda": (0, 6),
    "p_zero": (0, 0.5),
}

MODEL_NAME = "hetero3"
PARAMETERS = ("mu_lambda", "sigma_lambda", "p_zero")

# Plot style knobs
GROUND_TRUTH_COLOR = "tab:orange"
GROUND_TRUTH_LINESTYLE = "--"
GROUND_TRUTH_LINEWIDTH = 2.0

PAIR_XY_TITLE_SIZE = 50
PAIR_TICK_FONT_SIZE = 34
PAIR_LEGEND_FONT_SIZE = 34

HDI_XY_TITLE_SIZE = 44
HDI_TICK_FONT_SIZE = 34
HDI_LEGEND_FONT_SIZE = 34


### Posterior Distributions Across Synthetic Dataset Sizes

In [30]:
def apply_param_ticks(
    ax,
    *,
    xparam: Optional[str] = None,
    yparam: Optional[str] = None,
    param_ticks: Optional[Dict[str, Sequence[float]]] = None,
    param_ticklabels: Optional[Dict[str, Sequence[str]]] = None,
) -> None:
    if not param_ticks:
        return

    if xparam in param_ticks:
        ticks = list(param_ticks[xparam])
        ax.set_xticks(ticks)
        if param_ticklabels and xparam in param_ticklabels:
            ax.set_xticklabels(list(param_ticklabels[xparam]))

    if yparam in param_ticks:
        ticks = list(param_ticks[yparam])
        ax.set_yticks(ticks)
        if param_ticklabels and yparam in param_ticklabels:
            ax.set_yticklabels(list(param_ticklabels[yparam]))

def plot_posteriors(
    idatas: Sequence[Tuple[str, az.InferenceData]],
    *,
    ground_truth: Optional[Dict[str, Dict[str, float]]] = None,
    parameters: Sequence[str],
    parameter_display: Optional[Dict[str, str]] = None,
    show_legend: bool = True,
    hdi_prob: float = 0.95,
    sample_size: int = 200000,
    save_path: str | Path = "posteriors",
    cmap_name: str = "inferno",
    colors: Optional[Sequence[str] | Dict[str, str]] = None,
    font_scale: float = 0.7,
    diagonal_style: str = "hist",
    marginal_style: str = "circle",
    seed: Optional[int] = None,
    dpi: int = 300,
    xlims: Optional[Dict[str, Tuple[float, float]]] = None,
    label_size: int = PAIR_XY_TITLE_SIZE,
    tick_size: int = PAIR_TICK_FONT_SIZE,
    legend_size: int = PAIR_LEGEND_FONT_SIZE,
    ground_truth_color: str = GROUND_TRUTH_COLOR,
    ground_truth_linestyle: str = GROUND_TRUTH_LINESTYLE,
    ground_truth_linewidth: float = GROUND_TRUTH_LINEWIDTH,
    ground_truth_style: str = "lines",
    ground_truth_marker: str = "*",
    ground_truth_markersize: float = 24,
    param_ticks: Optional[Dict[str, Sequence[float]]] = None,
    param_ticklabels: Optional[Dict[str, Sequence[str]]] = None,
) -> None:
    sns.set_context("talk", font_scale=float(font_scale))
    labels = [label for label, _idata in idatas]
    if ground_truth_style not in {"lines", "star", "both"}:
        raise ValueError("ground_truth_style must be 'lines', 'star', or 'both'.")

    if colors is not None:
        if isinstance(colors, dict):
            plot_colors = [colors[label] for label in labels]
        else:
            plot_colors = list(colors)
            if len(plot_colors) != len(idatas):
                raise ValueError("colors must have the same length as idatas.")
    elif len(idatas) == 1:
        plot_colors = ["black"]
    else:
        cmap = plt.colormaps.get_cmap(str(cmap_name))
        plot_colors = cmap(np.linspace(0.5, 0.95, len(idatas)))
    rng = np.random.default_rng(seed)

    params = [str(p) for p in parameters]
    if parameter_display is None:
        parameter_display = {p: p for p in params}

    def _posterior_vals(posterior, name: str) -> np.ndarray:
        if name in posterior:
            vals = posterior[name].stack(sample=("chain", "draw")).values.ravel()
            vals = np.asarray(vals, dtype=float)
            return vals[np.isfinite(vals)]
        return np.array([], dtype=float)

    label_to_df: Dict[str, pd.DataFrame] = {}
    for label, idata in idatas:
        posterior = idata.posterior
        df = pd.DataFrame()
        for p in params:
            vals = _posterior_vals(posterior, p)
            if vals.size == 0:
                continue
            if vals.size > int(sample_size):
                idx = rng.choice(vals.size, int(sample_size), replace=False)
                vals = vals[idx]
            df[p] = vals
        if not df.empty:
            df["label"] = label
            label_to_df[label] = df

    def _robust_limits(all_vals: np.ndarray) -> Optional[Tuple[float, float]]:
        all_vals = np.asarray(all_vals, dtype=float)
        all_vals = all_vals[np.isfinite(all_vals)]
        if all_vals.size == 0:
            return None
        lo = float(np.quantile(all_vals, 0.005))
        hi = float(np.quantile(all_vals, 0.995))
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            lo = float(np.min(all_vals))
            hi = float(np.max(all_vals))
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            return None
        pad = 0.03 * (hi - lo)
        return lo - pad, hi + pad

    param_xlims: Dict[str, Tuple[float, float]] = {}
    for p in params:
        cols: List[np.ndarray] = []
        for _lab, _df in label_to_df.items():
            if p in _df.columns:
                arr = np.asarray(_df[p].values, dtype=float)
                arr = arr[np.isfinite(arr)]
                if arr.size:
                    cols.append(arr)
        if cols:
            lims = _robust_limits(np.concatenate(cols))
            if lims is not None:
                param_xlims[p] = lims

    if xlims:
        for k, v in xlims.items():
            if k in params and v is not None:
                param_xlims[k] = (float(v[0]), float(v[1]))

    npar = len(params)
    fig = plt.figure(figsize=(6 * npar, 6 * npar), dpi=int(dpi))
    fig.patch.set_alpha(0.0)
    gs = gridspec.GridSpec(npar, npar, wspace=0.38, hspace=0.38)
    gaxes = np.empty((npar, npar), dtype=object)

    truth_legend_drawn = False

    for irow, rowpar in enumerate(params):
        for icol, colpar in enumerate(params):
            ax = plt.subplot(gs[irow, icol])
            gaxes[irow, icol] = ax
            ax.set_facecolor("none")
            ax.xaxis.set_minor_locator(NullLocator())
            ax.yaxis.set_minor_locator(NullLocator())

            if icol > irow:
                ax.axis("off")
                continue

            for color, (label, df) in zip(plot_colors, label_to_df.items()):
                if icol == irow:
                    if rowpar not in df.columns:
                        continue
                    vals = df[rowpar].dropna().values
                    if vals.size == 0:
                        continue

                    if diagonal_style == "kde":
                        sns.kdeplot(
                            vals,
                            ax=ax,
                            fill=True,
                            color=color,
                            alpha=0.2,
                            linewidth=1.5,
                            label=(label if (bool(show_legend) and irow == 0) else None),
                        )
                    else:
                        sns.histplot(vals, bins=30, stat="density", kde=False, ax=ax, color=color, alpha=0.18, element="step", fill=True)
                        sns.histplot(
                            vals,
                            bins=30,
                            stat="density",
                            kde=False,
                            ax=ax,
                            color=color,
                            alpha=1.0,
                            element="step",
                            fill=False,
                            linewidth=1.8,
                            label=(label if (bool(show_legend) and irow == 0) else None),
                        )

                    try:
                        lo, hi = az.hdi(vals, hdi_prob=float(hdi_prob))
                        ax.axvspan(float(lo), float(hi), color=color, alpha=0.1, linewidth=0)
                    except Exception:
                        pass

                    if (
                        ground_truth is not None
                        and ground_truth_style in {"lines", "both"}
                        and label in ground_truth
                        and rowpar in ground_truth[label]
                    ):
                        truth_label = "Ground truth" if (bool(show_legend) and not truth_legend_drawn) else None
                        ax.axvline(
                            float(ground_truth[label][rowpar]),
                            color=ground_truth_color,
                            linestyle=ground_truth_linestyle,
                            linewidth=ground_truth_linewidth,
                            label=truth_label,
                            zorder=1000,
                        )
                        truth_legend_drawn = truth_legend_drawn or truth_label is not None

                    ax.grid(alpha=0.2)
                else:
                    if (colpar not in df.columns) or (rowpar not in df.columns):
                        continue
                    if marginal_style == "circle":
                        sns.kdeplot(x=df[colpar], y=df[rowpar], ax=ax, fill=False, color=color, alpha=0.6, levels=5, linewidths=1.2)
                    else:
                        sns.histplot(x=df[colpar], y=df[rowpar], bins=60, pthresh=0.01, cmap=str(cmap_name), cbar=False, ax=ax)

            if icol != irow:
                ax.grid(alpha=0.3)
                if ground_truth is not None:
                    truth_points_drawn = set()
                    for label, _df in label_to_df.items():
                        gt = ground_truth.get(label)
                        if gt and (colpar in gt) and (rowpar in gt):
                            truth_x = float(gt[colpar])
                            truth_y = float(gt[rowpar])
                            truth_key = (truth_x, truth_y)
                            if truth_key in truth_points_drawn:
                                continue
                            truth_points_drawn.add(truth_key)
                            truth_label = "Ground truth" if (bool(show_legend) and not truth_legend_drawn) else None
                            if ground_truth_style in {"lines", "both"}:
                                ax.axvline(
                                    truth_x,
                                    color=ground_truth_color,
                                    linestyle=ground_truth_linestyle,
                                    linewidth=ground_truth_linewidth,
                                    label=truth_label,
                                    zorder=1000,
                                )
                                ax.axhline(
                                    truth_y,
                                    color=ground_truth_color,
                                    linestyle=ground_truth_linestyle,
                                    linewidth=ground_truth_linewidth,
                                    zorder=1000,
                                )
                            if ground_truth_style in {"star", "both"}:
                                ax.scatter(
                                    [truth_x],
                                    [truth_y],
                                    marker=ground_truth_marker,
                                    s=float(ground_truth_markersize) ** 2,
                                    color=ground_truth_color,
                                    edgecolors="black",
                                    linewidths=0.8,
                                    label=truth_label,
                                    zorder=1001,
                                )
                            truth_legend_drawn = truth_legend_drawn or truth_label is not None

            if icol == irow:
                ax.set_xlabel(parameter_display.get(rowpar, rowpar), fontsize=label_size)
                ax.set_ylabel("Density", fontsize=label_size)
            else:
                ax.set_xlabel(parameter_display.get(colpar, colpar), fontsize=label_size)
                ax.set_ylabel(parameter_display.get(rowpar, rowpar), fontsize=label_size)
            ax.tick_params(axis="both", which="major", labelsize=tick_size)
            # ax.tick_params(axis="both", which="minor", labelsize=tick_size)
            if icol == irow:
                xpar = rowpar
                if xpar in param_xlims:
                    ax.set_xlim(*param_xlims[xpar])
                apply_param_ticks(
                    ax,
                    xparam=rowpar,
                    param_ticks=param_ticks,
                    param_ticklabels=param_ticklabels,
                )
            else:
                if colpar in param_xlims:
                    ax.set_xlim(*param_xlims[colpar])
                if rowpar in param_xlims:
                    ax.set_ylim(*param_xlims[rowpar])
                apply_param_ticks(
                    ax,
                    xparam=colpar,
                    yparam=rowpar,
                    param_ticks=param_ticks,
                    param_ticklabels=param_ticklabels,
                )

            ax.margins(0.05)

    if bool(show_legend):
        legend_items = {}
        for ax in gaxes.ravel():
            if ax is None:
                continue
            handles, leg_labels = ax.get_legend_handles_labels()
            for handle, leg_label in zip(handles, leg_labels):
                if leg_label and not leg_label.startswith("_"):
                    legend_items.setdefault(leg_label, handle)
        if legend_items:
            fig.legend(list(legend_items.values()), list(legend_items.keys()), loc="center", bbox_to_anchor=(0.85, 0.85), frameon=True, edgecolor="black", fontsize=legend_size)
    
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path.with_suffix(".svg"), dpi=int(dpi), bbox_inches="tight", transparent=True)
    plt.close(fig)
    # print("Saved joint posterior plot:", str(save_path.with_suffix(".pdf")))


In [3]:
def load_config(n_cells: int, results_root: Path = RESULTS_DIR) -> dict:
    path = results_root / str(n_cells) / "config_ZI-gamma.json"
    return pd.read_json(path, typ="series").to_dict()

def load_posterior(n_cells: int, results_root: Path = RESULTS_DIR) -> az.InferenceData:
    path = results_root / str(n_cells) / "posterior_ZI-gamma_hetero3_smc.nc"
    return az.from_netcdf(path)

def load_logml(n_cells: int, results_root: Path = RESULTS_DIR) -> pd.DataFrame:
    path = results_root / str(n_cells) / "log_marginal_likelihood_ZI-gamma_hetero3_smc.csv"
    return pd.read_csv(path)


sample_sizes = sorted(
    int(p.name)
    for p in RESULTS_DIR.iterdir()
    if p.is_dir() and p.name.isdigit()
)

results = {
    n: {
        "config": load_config(n),
        "idata": load_posterior(n),
        "logml": load_logml(n),
    }
    for n in sample_sizes
}

CONSISTENCY_KEYS = (
    "T",
    "gt_mu_lambda",
    "gt_sigma_lambda",
    "gt_p0_lambda",
    "seed",
    "smc_particles",
    "std_prior_factor",
    "lambda_prior_bounds",
    "threshold",
    "correlation_threshold",
)

def _config_value(config: dict, key: str):
    value = config.get(key)
    return tuple(value) if isinstance(value, list) else value

if results:
    reference_n = sample_sizes[0]
    reference_config = results[reference_n]["config"]
    for n, item in results.items():
        config = item["config"]
        mismatches = [
            key
            for key in CONSISTENCY_KEYS
            if _config_value(config, key) != _config_value(reference_config, key)
        ]
        if mismatches:
            raise ValueError(f"Config mismatch for n={n}: {mismatches}. Rerun all sample sizes into one clean RESULTS_DIR.")

print("sample size (number of in-silico NK cells) analysed:", "\n", sample_sizes)

sample size (number of in-silico NK cells) analysed: 
 [10, 20, 30, 50, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]


In [ ]:
FIG_DIR = Path("../figures") / RUN_NAME
FIG_DIR.mkdir(parents=True, exist_ok=True)

ground_truth = {}

mu_lambda_ticks = [0, 4, 8]
sigma_lambda_ticks = [0, 3, 6]
phi_ticks = [0, 0.2, 0.4,]


for n in tqdm(sample_sizes):
    config = load_config(n)
    idata = load_posterior(n)

    label = f"n={n}"

    ground_truth[label] = {
        "mu_lambda": config["gt_mu_lambda"],
        "sigma_lambda": config["gt_sigma_lambda"],
        "p_zero": config["gt_p0_lambda"],
    }

    plot_posteriors(
        [(label, idata)],
        parameters=PARAMETERS,
        ground_truth=ground_truth,
        parameter_display=PARAM_LABELS,
        xlims=PARAM_XLIMS,
        show_legend=False,
        save_path=FIG_DIR / f"posterior_{label}.pdf",
        diagonal_style="hist",
        marginal_style="circle",
        label_size=PAIR_XY_TITLE_SIZE,
        tick_size=PAIR_TICK_FONT_SIZE,
        seed=2026,
        param_ticks={
            "mu_lambda": mu_lambda_ticks,
            "sigma_lambda": sigma_lambda_ticks,
            "p_zero": phi_ticks},
        param_ticklabels={
            "mu_lambda": ["0", "4", "8"],
            "sigma_lambda": ["0", "3", "6"],
            "p_zero": ["0", "0.2", "0.4"]
        },
    )

 29%|██▊       | 4/14 [01:19<03:18, 19.83s/it]

### Variation in Posterior HDIs Across Sample Sizes

In [37]:
LABEL_SIZE = HDI_XY_TITLE_SIZE
TICK_SIZE = HDI_TICK_FONT_SIZE
LEGEND_SIZE = HDI_LEGEND_FONT_SIZE
TICK_NBINS = 4


def param_display_label(param: str) -> str:
    return PARAM_LABELS.get(param, param)


def posterior_values(idata: az.InferenceData, param: str) -> np.ndarray:
    if param not in idata.posterior:
        return np.array([], dtype=float)

    vals = idata.posterior[param].stack(sample=("chain", "draw")).values.ravel()
    vals = np.asarray(vals, dtype=float)
    return vals[np.isfinite(vals)]


def sample_size_ticks(sample_sizes: Sequence[int], xscale: str) -> list[int]:
    sizes = sorted(int(x) for x in sample_sizes)
    lo, hi = sizes[0], sizes[-1]

    if xscale == "log":
        ticks = []
        start_decade = int(np.floor(np.log10(lo)))
        end_decade = int(np.ceil(np.log10(hi)))
        for decade in range(start_decade, end_decade + 1):
            for multiplier in (1, 2, 5):
                tick = int(multiplier * (10 ** decade))
                if lo <= tick <= hi:
                    ticks.append(tick)
    else:
        locator = MaxNLocator(nbins=6, integer=True)
        ticks = [int(t) for t in locator.tick_values(lo, hi) if lo <= t <= hi]

    ticks.extend([lo, hi])
    return sorted(set(ticks))


def plot_hdi_summary(
    sample_sizes: Sequence[int],
    output_path: Path,
    *,
    results_root: Path = RESULTS_DIR,
    dpi: int = 350,
    hdi_prob: float = 0.95,
    cmap_name: str = "YlGnBu",
    cmap_min: float = 0.6,
    cmap_max: float = 0.95,
    xlim: Optional[Tuple[float, float]] = None,
    xscale: str = "linear",
) -> None:
    sample_sizes = sorted(int(x) for x in sample_sizes)
    if xscale not in {"linear", "log"}:
        raise ValueError("xscale must be 'linear' or 'log'.")
    if xscale == "log" and min(sample_sizes) <= 0:
        raise ValueError("Log-scale sample sizes must be positive.")

    rows = []

    for n_cells in sample_sizes:
        idata = load_posterior(n_cells, results_root)

        for param in PARAMETERS:
            vals = posterior_values(idata, param)
            if vals.size == 0:
                continue

            lo, hi = az.hdi(vals, hdi_prob=float(hdi_prob))
            rows.append(
                {
                    "n_cells": int(n_cells),
                    "param": param,
                    "hdi_low": float(lo),
                    "hdi_high": float(hi),
                }
            )

    if not rows:
        print("No posterior samples found.")
        return

    df = pd.DataFrame(rows).sort_values("n_cells")

    config0 = load_config(int(sample_sizes[0]), results_root)
    ground_truth = {
        "mu_lambda": float(config0["gt_mu_lambda"]),
        "sigma_lambda": float(config0["gt_sigma_lambda"]),
        "p_zero": float(config0["gt_p0_lambda"]),
    }

    cmap = plt.colormaps.get_cmap(str(cmap_name))
    colors = cmap(np.linspace(float(cmap_min), float(cmap_max), len(PARAMETERS)))
    param_to_color = {param: colors[i] for i, param in enumerate(PARAMETERS)}

    fig, axes = plt.subplots(
        len(PARAMETERS),
        1,
        figsize=(16, 16),
        dpi=int(dpi),
        sharex=True,
    )

    if len(PARAMETERS) == 1:
        axes = [axes]

    if xlim is None:
        if xscale == "log":
            xlim = ( min(sample_sizes), max(sample_sizes))
        else:
            pad = 0.01 * (max(sample_sizes) - min(sample_sizes)) if len(sample_sizes) > 1 else 1.0
            xlim = (min(sample_sizes) - pad, max(sample_sizes) + pad)

    for ax, param in zip(axes, PARAMETERS):
        ax.set_facecolor("none")

        sub = df[df["param"] == param].sort_values("n_cells")
        if sub.empty:
            ax.axis("off")
            continue

        color = param_to_color[param]

        x = sub["n_cells"].to_numpy(dtype=float)
        lo = sub["hdi_low"].to_numpy(dtype=float)
        hi = sub["hdi_high"].to_numpy(dtype=float)

        ax.fill_between(
            x,
            lo,
            hi,
            color=color,
            alpha=0.15,
            linewidth=0,
            label=f"{param_display_label(param)} {int(hdi_prob * 100)} \% HDI",
        )

        ax.plot(
            x,
            np.full_like(x, ground_truth[param], dtype=float),
            color=GROUND_TRUTH_COLOR,
            linestyle=GROUND_TRUTH_LINESTYLE,
            linewidth=GROUND_TRUTH_LINEWIDTH,
            label="Ground truth",
        )

        ax.grid(alpha=0.25)
        ax.set_xscale(xscale)
        ax.set_xlim(*xlim)
        ax.tick_params(axis="both", which="major", labelsize=TICK_SIZE)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=TICK_NBINS))
        ax.set_ylabel(param_display_label(param), fontsize=LABEL_SIZE)
        ax.legend(frameon=False, fontsize=LEGEND_SIZE, loc="upper right", handlelength=1.5, handleheight=1.5, labelspacing=0.3, borderpad=0.3)
        ax.xaxis.set_minor_locator(NullLocator())
        ax.yaxis.set_minor_locator(NullLocator())
        ax.minorticks_off()

    x_ticks = sample_size_ticks(sample_sizes, xscale)
    axes[-1].set_xlabel("Number of NK cells", fontsize=LABEL_SIZE)
    axes[-1].set_xticks(x_ticks)
    axes[-1].set_xticklabels([str(n) for n in x_ticks])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=int(dpi), bbox_inches="tight", transparent=True)
    plt.close(fig)

    print(f"Saved: {output_path}")

<>:136: SyntaxWarning: invalid escape sequence '\%'
<>:136: SyntaxWarning: invalid escape sequence '\%'
/var/folders/rr/16wv6ts1785fz6_2qhr5bc2m0000gn/T/ipykernel_58711/3821713101.py:136: SyntaxWarning: invalid escape sequence '\%'
  label=f"{param_display_label(param)} {int(hdi_prob * 100)} \% HDI",


In [38]:
FIG_DIR = Path("../figures") / RUN_NAME
FIG_DIR.mkdir(parents=True, exist_ok=True)

sample_sizes = sorted(
    int(p.name)
    for p in RESULTS_DIR.iterdir()
    if p.is_dir() and p.name.isdigit()
)

for xscale in ("linear", "log"):
    plot_hdi_summary(
        sample_sizes=sample_sizes,
        output_path=FIG_DIR / f"hetero3_hdi_density_by_cell_number_{xscale}.svg",
        results_root=RESULTS_DIR,
        hdi_prob=0.95,
        xscale=xscale,
    )

Saved: ../figures/part_1_runs/mu4.0_sigma3.0_p00.2_seed26/hetero3_hdi_density_by_cell_number_linear.svg
Saved: ../figures/part_1_runs/mu4.0_sigma3.0_p00.2_seed26/hetero3_hdi_density_by_cell_number_log.svg


### Posterior Comparison for 50, 500, and 1000 Cells

In [36]:
comparison_sizes = [50, 500, 1000]

comparison_idatas = []
comparison_ground_truth = {}

for n in comparison_sizes:
    label = f"{n} cells"
    config = load_config(n)
    idata = load_posterior(n)

    comparison_idatas.append((label, idata))
    comparison_ground_truth[label] = {
        "mu_lambda": config["gt_mu_lambda"],
        "sigma_lambda": config["gt_sigma_lambda"],
        "p_zero": config["gt_p0_lambda"],
    }


plot_posteriors(
    comparison_idatas,
    parameters=PARAMETERS,
    ground_truth=comparison_ground_truth,
    parameter_display=PARAM_LABELS,
    xlims=PARAM_XLIMS,
    show_legend=True,
    save_path=FIG_DIR / "posterior_comparison_n50_n500_n1000",
    diagonal_style="hist",
    marginal_style="circle",
    cmap_name="Blues",
    label_size=PAIR_XY_TITLE_SIZE,
    tick_size=PAIR_TICK_FONT_SIZE,
    legend_size=PAIR_LEGEND_FONT_SIZE,
    seed=2026,
    param_ticks={
        "mu_lambda": [0, 4, 8],
        "sigma_lambda": [0, 3, 6],
        "p_zero": [0, 0.2, 0.4],
    },
    param_ticklabels={
        "mu_lambda": ["0", "4", "8"],
        "sigma_lambda": ["0", "3", "6"],
        "p_zero": ["0", "0.2", "0.4"],
    },
)